In [1]:
# Install + imports
!pip install opencv-python-headless

import cv2
import numpy as np
from google.colab import files
from google.colab.patches import cv2_imshow

# Upload video
uploaded = files.upload()
video_path = list(uploaded.keys())[0]

# Video capture
cap = cv2.VideoCapture(video_path)

ret, frame1 = cap.read()
ret, frame2 = cap.read()

# Optional: get frame size for saving
height, width, _ = frame1.shape
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter('output.avi', fourcc, 20.0, (width, height))

while ret:
    # Convert to grayscale
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)

    # Blur
    gray1 = cv2.GaussianBlur(gray1, (5,5), 0)
    gray2 = cv2.GaussianBlur(gray2, (5,5), 0)

    # Frame difference
    diff = cv2.absdiff(gray1, gray2)

    # Threshold
    _, thresh = cv2.threshold(diff, 25, 255, cv2.THRESH_BINARY)

    # Dilate
    dilated = cv2.dilate(thresh, None, iterations=2)

    # Contours
    contours, _ = cv2.findContours(dilated, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        if cv2.contourArea(contour) < 1000:
            continue

        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(frame1, (x,y), (x+w,y+h), (0,255,0), 2)

    # Show frame
    cv2_imshow(frame1)

    # Save output
    out.write(frame1)

    # Next frames
    frame1 = frame2
    ret, frame2 = cap.read()

cap.release()
out.release()

print("Processing complete. Output saved as output.avi")

Output hidden; open in https://colab.research.google.com to view.